In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import subprocess

In [2]:
sys.path.append("../../../training_data")

In [3]:
from utils.utils import Cif

In [4]:
with open("../../../training_data/7.Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pd.read_pickle(f)

len(extras_featuresd), extras_featuresd

(9,
 {'7gqu':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       7gqu               1             A           12            A   
  1       7gqu               1             A           13            A   
  2       7gqu               1             A           14            A   
  3       7gqu               1             A           15            A   
  4       7gqu               1             A           16            A   
  ..       ...             ...           ...          ...          ...   
  414     7gqu               1             A          426            A   
  415     7gqu               1             A          427            A   
  416     7gqu               1             A          428            A   
  417     7gqu               1             A          429            A   
  418     7gqu               1             A          430            A   
  
                       

# Make predictions

Edits throughout to fix:
- Hardcoded paths
- Pass locations of ProtT5 and the nr database

In [5]:
for pdb, feats in extras_featuresd.items():
    if pdb == "8aq6": continue
    path = Path("AlloFusion/Case Study").resolve()
    path.mkdir(exist_ok = True)
    try:
        outdir = Path(pdb)
        if not outdir.exists():
            chain = feats[('Residues', 'auth_asym_id')].unique().item()
            
            origpdbf = path.parents[2] / "structures" / f"{pdb.lower()}.pdb"
    
            seq = (
                pd.DataFrame(
                    Cif(pdb, origpdbf.with_suffix(".cif")).cif.data["_entity_poly"], dtype=str
                )
                .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
                ["pdbx_seq_one_letter_code_can"].item()
                .replace("\n", "")
            )
            
            pdbf = path / f"{pdb}.pdb"
            if not pdbf.exists():
                pdbf.symlink_to(origpdbf)
        
            subprocess.run(f"python AlloFusionMain.py --PDBID {pdb} --CHAIN {chain} --SEQ {seq} --huggingface_dir /data/fnerin/huggingface --nr_database /data/fnerin/nr_database/nr", cwd="AlloFusion", shell=True, check=True)

            path.rename(outdir)
            
    except Exception as e:
        print(f"ERROR: ", pdb)
        print(e)
        path.rename(f"{pdb}_error")
        continue

In [9]:
for pdb, feats in extras_featuresd.items():
    if pdb != "8aq6": continue
    
    path = Path("AlloFusion/Case Study").resolve()
    path.mkdir(exist_ok = True)
    
    for chain in ("G", "H"):
        try:
            outdir = Path(f"{pdb}_{chain}")
            if not outdir.exists():
                # chain = feats[('Residues', 'auth_asym_id')].unique().item()
                
                origpdbf = path.parents[2] / "structures" / f"{pdb.lower()}.pdb"
        
                seq = (
                    pd.DataFrame(
                        Cif(pdb, origpdbf.with_suffix(".cif")).cif.data["_entity_poly"], dtype=str
                    )
                    .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
                    ["pdbx_seq_one_letter_code_can"].item()
                    .replace("\n", "")
                )
                
                pdbf = path / f"{pdb}.pdb"
                if not pdbf.exists():
                    pdbf.symlink_to(origpdbf)
            
                subprocess.run(f"python AlloFusionMain.py --PDBID {pdb} --CHAIN {chain} --SEQ {seq} --huggingface_dir /data/fnerin/huggingface --nr_database /data/fnerin/nr_database/nr", cwd="AlloFusion", shell=True, check=True)
        
                path.rename(outdir)
                
        except Exception as e:
            print(f"ERROR: ", pdb)
            print(e)
            path.rename(f"{pdb}_error")
            continue

2025-12-09 19:14:38.630441: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 19:14:38.752973: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and

Embedding done!
PSSM done!
Bio done!
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step


# Process

In [10]:
results = {}

for pdb, feats in extras_featuresd.items():
    if pdb == "8aq6": continue
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    resf = f"{pdb}/{pdb}_allosteric_residues.txt"
    if os.path.isfile(resf):
        with open(resf) as f:
            txt = f.read()
        chain = txt.split("Chain", 1)[1].strip().split()[0]
        resids = [x for x in txt.split("resid", 1)[1].replace("(", "").replace(")", "").replace(",", " ").split() if x.isdigit()]   

        results[pdb.lower()] = {"pocket": {"residues": (
            pd.DataFrame({"auth_asym_id": [chain]*len(resids), "auth_seq_id": resids}, dtype=str)
            .merge(Cif(pdb, f"../structures/{pdb}.cif").residues)
            [["auth_asym_id", "auth_seq_id"]]
        )}}

len(results), results

(8,
 {'7gqu': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         577
    1            A         581
    2            A         713
    3            A         721
    4            A         730
    5            A         731
    6            A         854
    7            A         888
    8            A         891}},
  '7yg5': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         381
    1            A        1354
    2            A        1397}},
  '8f4s': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A        6878
    1            A        6880
    2            A        6968
    3            A        7000}},
  '8jp0': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         486
    1            A         809}},
  '8qni': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         132
    1            A         141
    2            A         143
    3            A         2

In [11]:
results["8aq6"] = {}

for pdb, feats in extras_featuresd.items():
    if pdb != "8aq6": continue
    chains = feats[('Residues', 'auth_asym_id')].unique()
    for chain in chains:
        resf = f"{pdb}_{chain}/{pdb}_allosteric_residues.txt"
        if os.path.isfile(resf):
            with open(resf) as f:
                txt = f.read()
            chain = txt.split("Chain", 1)[1].strip().split()[0]
            resids = [x for x in txt.split("resid", 1)[1].replace("(", "").replace(")", "").replace(",", " ").split() if x.isdigit()]   

            results["8aq6"][f"pocket_{chain}"] = {"residues": (
                pd.DataFrame({"auth_asym_id": [chain]*len(resids), "auth_seq_id": resids}, dtype=str)
                .merge(Cif(pdb, f"../structures/{pdb}.cif").residues)
                [["auth_asym_id", "auth_seq_id"]]
            )}

In [12]:
len(results), results

(9,
 {'7gqu': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         577
    1            A         581
    2            A         713
    3            A         721
    4            A         730
    5            A         731
    6            A         854
    7            A         888
    8            A         891}},
  '7yg5': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         381
    1            A        1354
    2            A        1397}},
  '8f4s': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A        6878
    1            A        6880
    2            A        6968
    3            A        7000}},
  '8jp0': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         486
    1            A         809}},
  '8qni': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         132
    1            A         141
    2            A         143
    3            A         2

In [13]:
pd.to_pickle(results, "allofusion_results.pkl")